<a href="https://colab.research.google.com/github/jamesemcnally/critical-listener/blob/main/recommender_2_with_name_masking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install anthropic -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Imports and GPU check.
import pandas as pd
import numpy as np
from itertools import combinations
import random
import torch

print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


GPU available: True
Device: NVIDIA L4


### Loading masked cleaned dataset from Colab

In [ ]:
df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/merged_dataset_masked.parquet")
print(f"Loaded: {df.shape[0]:,} reviews")
print(df.columns.tolist())


Loaded: 48,189 reviews
['dataset', 'review_id', 'entity_id', 'text', 'rating', 'album', 'artist', 'reviewer_name', 'reviewer_id', 'reviewer_type', 'reviewer_karma', 'published_on', 'source', 'source_url', 'cleaned_text', 'word_count', 'char_count', 'sentence_count', 'sentiment_score', 'sentiment_category', 'genre', 'ra_recommends', 'artist_norm', 'album_norm', 'input_with_prefix', 'input_no_prefix', 'personnel', 'cleaned_text_masked']


## Masked Embeddings

This notebook re-embeds the review corpus using `cleaned_text_masked` — a version of each review where band members, credited artists, and producers have been replaced with `[MASKED]`. The goal is to reduce identity leakage: the model should find semantic similarity based on critical language and themes, not because two reviews mention the same artist names.

Compare results against the original embeddings (`nomic_with_prefix.npy`) to measure the effect of masking.


In [ ]:
# Create masked model inputs with and without artist/album prefix.
# Uses cleaned_text_masked so personnel names are replaced with [MASKED].
df["input_masked_with_prefix"] = (
    "Artist: " + df["artist"].fillna("Unknown") + ". " +
    "Album: "  + df["album"].fillna("Unknown")  + ". " +
    df["cleaned_text_masked"]
)
df["input_masked_no_prefix"] = df["cleaned_text_masked"]

print(df["input_masked_with_prefix"].iloc[10][:300])


Artist: AC/DC. Album: Back in Black. Back in Black is one of those albums where you instantly understand why it became legendary. The riffs are huge, the rhythm section is locked in, and the whole thing moves with confidence and swagger. Not every song hit me equally, but the highs are untouchable. 


In [ ]:
# Evaluation pairs, sample, and retrieval metric function.
# Same as first notebook — reused here for model comparison.
random.seed(42)

eval_pairs = []
keyed = df.dropna(subset=["artist_norm", "album_norm"])
for _, group in keyed.groupby(["artist_norm", "album_norm"]):
    if group["dataset"].nunique() > 1:
        for i, j in combinations(group.index.tolist(), 2):
            if df.loc[i, "dataset"] != df.loc[j, "dataset"]:
                eval_pairs.append((i, j))

eval_pairs_sample = random.sample(eval_pairs, 500)
results = {}

def evaluate_retrieval(get_sims_fn, eval_pairs, ks=(1, 5, 10)):
    rr, hits = [], {k: [] for k in ks}
    for q_idx, t_idx in eval_pairs:
        sims = get_sims_fn(q_idx).copy()
        sims[q_idx] = -np.inf
        target_sim = sims[t_idx]
        same_album = df[
            (df["artist_norm"] == df.loc[q_idx, "artist_norm"]) &
            (df["album_norm"]  == df.loc[q_idx, "album_norm"])
        ].index
        sims[same_album] = -np.inf
        sims[t_idx] = target_sim
        ranked = np.argsort(sims)[::-1]
        rank = int(np.where(ranked == t_idx)[0][0]) + 1
        rr.append(1.0 / rank)
        for k in ks:
            hits[k].append(int(rank <= k))
    return {"MRR": np.mean(rr), **{f"Recall@{k}": np.mean(hits[k]) for k in ks}}

print(f"Eval pairs: {len(eval_pairs):,} | Sample: {len(eval_pairs_sample)}")


Eval pairs: 3,653 | Sample: 500


In [ ]:
# Embed masked reviews with Nomic (8,192-token long-context model).
# Saves embeddings to Drive immediately after each run in case of disconnect.
from sentence_transformers import SentenceTransformer

model_nomic = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

for input_col, label, save_name in [
    ("input_masked_with_prefix", "Nomic masked (with prefix)", "nomic_masked_with_prefix"),
    ("input_masked_no_prefix",   "Nomic masked (no prefix)",   "nomic_masked_no_prefix")
]:
    embeddings = model_nomic.encode(
        df[input_col].tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    np.save(f"/content/drive/MyDrive/Colab Notebooks/{save_name}.npy", embeddings)
    print(f"Saved {save_name}.npy to Drive")

    def nomic_sims(idx, embeddings=embeddings):
        return embeddings @ embeddings[idx]

    results[label] = evaluate_retrieval(nomic_sims, eval_pairs_sample)
    print(f"\n{label}")
    print("-" * 40)
    for metric, val in results[label].items():
        print(f"  {metric}: {val:.4f}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/1506 [00:00<?, ?it/s]

[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


Saved nomic_masked_with_prefix.npy to Drive

Nomic masked (with prefix)
----------------------------------------
  MRR: 0.9335
  Recall@1: 0.8980
  Recall@5: 0.9820
  Recall@10: 0.9900


Batches:   0%|          | 0/1506 [00:00<?, ?it/s]

Saved nomic_masked_no_prefix.npy to Drive

Nomic masked (no prefix)
----------------------------------------
  MRR: 0.6307
  Recall@1: 0.5540
  Recall@5: 0.7260
  Recall@10: 0.7680


In [ ]:
# Load masked Nomic embeddings — used for the masked recommender.
embeddings_masked = np.load("/content/drive/MyDrive/Colab Notebooks/nomic_masked_with_prefix.npy")
print(f"Loaded: {embeddings_masked.shape}")


Loaded: (48189, 768)


In [ ]:
# Masked recommender with keyword overlap preview.
# Shows distinctive shared words between query and recommended review
# as a quick pre-explainer before committing to a full API call.
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Fit TF-IDF on full corpus once — used to identify distinctive words
tfidf_preview = TfidfVectorizer(max_features=50_000, min_df=2, stop_words="english")
tfidf_preview.fit(df["cleaned_text"].dropna())
tfidf_vocab = tfidf_preview.get_feature_names_out()

def keyword_overlap(q_text, r_text, top_n=8):
    vecs = tfidf_preview.transform([q_text, r_text]).toarray()
    # Words that score highly in BOTH reviews
    shared = np.minimum(vecs[0], vecs[1])
    top_idx = np.argsort(shared)[::-1][:top_n]
    return [tfidf_vocab[i] for i in top_idx if shared[i] > 0]

def recommend_masked(query_album, query_artist=None, k=5, filter_same_artist=True):
    mask = df["album_norm"] == query_album.lower().strip()
    if query_artist:
        mask &= df["artist_norm"] == query_artist.lower().strip()

    query_reviews = df[mask]
    if len(query_reviews) == 0:
        print(f"Not found: '{query_album}'")
        return None

    query_artist_norm = query_reviews.iloc[0]["artist_norm"]
    q_text = query_reviews.iloc[0]["cleaned_text"]
    query_embedding = embeddings_masked[query_reviews.index].mean(axis=0)
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    sims = embeddings_masked @ query_embedding
    sims[query_reviews.index] = -np.inf

    ranked = np.argsort(sims)[::-1]
    seen, results = set(), []
    for idx in ranked:
        artist_norm = df.loc[idx, "artist_norm"]
        if filter_same_artist and artist_norm == query_artist_norm:
            continue
        key = (artist_norm, df.loc[idx, "album_norm"])
        if key not in seen:
            seen.add(key)
            r_text = df.loc[idx, "cleaned_text"]
            results.append({
                "album":    df.loc[idx, "album"],
                "artist":   df.loc[idx, "artist"],
                "source":   df.loc[idx, "dataset"],
                "score":    round(float(sims[idx]), 4),
                "keywords": keyword_overlap(q_text, r_text),
                "snippet":  r_text[:200]
            })
        if len(results) >= k:
            break

    for i, r in enumerate(results, 1):
        print(f"\n{i}. {r['artist']} — {r['album']} ({r['source']}, score: {r['score']})")
        print(f"   Keywords: {', '.join(r['keywords'])}")
        print(f"   {r['snippet']}...")

    return results

# Test on Charli XCX - Pop 2
recs = recommend_masked("pop 2", query_artist="charli xcx")



1. Icona Pop — The Iconic EP (pitchfork, score: 0.7841)
   Keywords: pop, charli, xcx, like, icona, love, robyn, spears
   In a pop landscape where turned-up-to-11 is the new quiet (thanks, Skrillex) and a handful of familiar faces dominate the charts, what does a group of newcomers have to do to turn heads? "I Love It", ...

2. Annie — Anniemal (pitchfork, score: 0.7833)
   Keywords: pop, like, song, world, love, electro, knee, jerk
   Depending on whom you ask-- and even in which country you live-- "pop" can mean a lot of different things. It's a word that's been demonized and marginalized, especially amongst stodgy and often conse...

3. Slayyyter — Troubled Paradise (pitchfork, score: 0.7826)
   Keywords: xcx, charli, pop, petras, mixtape, tweets, music, like
   In the world of 24-year-old glitch-pop artist Slayyyter, the Juicy Couture lockets sway to the booming bass, celebrities from 2004 are ancient deities, and when you sing along, you should sound like B...

4. A. G. Cook — 7

### Import explainer function

In [ ]:
# Simplified explainability: identify top shared qualities grounded in verbatim quotes.
# Honest about weak connections — if fewer than n genuine overlaps exist, shows fewer.
# Optional source parameter restricts to a specific dataset (e.g. "pitchfork").
import anthropic
import re
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

def explain_pair_simple(query_album, query_artist, rec_album, rec_artist, n=3, source=None):
    q_mask = (
        (df["album_norm"] == query_album.lower().strip()) &
        (df["artist_norm"] == query_artist.lower().strip())
    )
    r_mask = (
        (df["album_norm"] == rec_album.lower().strip()) &
        (df["artist_norm"] == rec_artist.lower().strip())
    )
    if source:
        q_mask &= df["dataset"] == source
        r_mask &= df["dataset"] == source

    q_reviews = df[q_mask]
    r_reviews = df[r_mask]

    if q_reviews.empty or r_reviews.empty:
        print("One or both albums not found in specified source.")
        return

    q_text   = q_reviews.iloc[0]["cleaned_text"]
    r_text   = r_reviews.iloc[0]["cleaned_text"]
    q_source = q_reviews.iloc[0]["dataset"]
    r_source = r_reviews.iloc[0]["dataset"]
    q_label  = f"{query_artist} — {query_album} ({q_source})"
    r_label  = f"{rec_artist} — {rec_album} ({r_source})"

    prompt = f"""You are analyzing two music reviews to find their most meaningful shared qualities.

Identify up to {n} significant qualities that both albums genuinely share, based ONLY on what is actually written in the reviews.
If fewer than {n} genuine connections exist, list only those that are real — do not pad with weak or generic observations.
If there are no meaningful connections, say so plainly.

For each shared quality:
- State it in one clear phrase
- Quote the specific language from each review that supports it (verbatim, in quotation marks, attributed to Review A or Review B)

Album A: {q_label}
Review A:
{q_text[:2000]}

Album B: {r_label}
Review B:
{r_text[:2000]}

Return a numbered list only. No introduction, no conclusion."""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    )

    print(f"\n{'='*60}")
    print(f"  {q_label}")
    print(f"  → {r_label}")
    print(f"{'='*60}")
    print(f"\n{response.content[0].text.strip()}")

### Trying out some of the pairs to see what the explainer produces.

In [ ]:
endochine_row = df[(df["artist_norm"] == "endochine") & (df["album_norm"] == "day two")].iloc[0]
radiohead_row = df[(df["artist_norm"] == "radiohead") & (df["album_norm"] == "kid a") & (df["dataset"] == "pitchfork")].iloc[0]

explain_pair_simple(
    radiohead_row["album_norm"], radiohead_row["artist_norm"],
    endochine_row["album_norm"], endochine_row["artist_norm"],
    source="pitchfork"
)



  radiohead — kid a (pitchfork)
  → endochine — day two (pitchfork)

1. **Emotional intensity and grandiose atmospherics**
   - Review A: "The human part of me wept in awe"
   - Review B: "the band shoots for epic anthems and sometimes come close to hitting the mark"

2. **Influence of Thom Yorke's vocal style**
   - Review A: "Thom Yorke slowly beat on a grand piano, singing, eyes closed, into his microphone"
   - Review B: "pinch their falsettos from Coldplay's Chris Martin" and "borrows almost every one of Yorke's signature moves, including those little slurs into dissonance he does so well"


In [ ]:
muse_row = df[(df["artist_norm"] == "muse") & (df["album_norm"] == "showbiz") & (df["dataset"] == "pitchfork")].iloc[0]
aero_row = df[(df["artist_norm"] == "aereogramme") & (df["album_norm"] == "sleep and release") & (df["dataset"] == "pitchfork")].iloc[0]

explain_pair_simple(
    radiohead_row["album_norm"], radiohead_row["artist_norm"],
    muse_row["album_norm"], muse_row["artist_norm"],
    source="pitchfork"
)

print("\n\n")

explain_pair_simple(
    radiohead_row["album_norm"], radiohead_row["artist_norm"],
    aero_row["album_norm"], aero_row["artist_norm"],
    source="pitchfork"
)



  radiohead — kid a (pitchfork)
  → muse — showbiz (pitchfork)

1. **Vocal performance as a distinctive artistic signature**
   - Review A: "Thom Yorke slowly beat on a grand piano, singing, eyes closed, into his microphone like he was trying to kiss around a big nose"
   - Review B: "Matthew Bellamy's falsetto-laden yelps, which sound so much like Thom Yorke they could pass as Andy Yorke"

2. **Ambitious artistic approach and influence**
   - Review A: "The trained critical part of me marked the similarity to Coltrane's 'Ole'" (indicating high-minded musical ambition)
   - Review B: "Muse expertly boil down Radiohead into punkish radio nuggets" and reference to Radiohead trying to "approach it like Miles Davis"




  radiohead — kid a (pitchfork)
  → aereogramme — sleep and release (pitchfork)

1. **Ambitious sonic experimentation across diverse influences**
   - Review A: "The trained critical part of me marked the similarity to Coltrane's 'Ole.'" and the description of material bei

In [ ]:
sophie_row = df[(df["artist_norm"] == "sophie") & (df["album_norm"] == "oil of every pearl's un-insides")].iloc[0]
charli_row = df[(df["artist_norm"] == "charli xcx") & (df["album_norm"] == "pop 2")].iloc[0]

explain_pair_simple(
    charli_row["album_norm"], charli_row["artist_norm"],
    sophie_row["album_norm"], sophie_row["artist_norm"]
)



  charli xcx — pop 2 (pitchfork)
  → sophie — oil of every pearl's un-insides (resident_advisor)

1. **Blending avant-garde experimental production with pop accessibility**
   - Review A: "she doubled down on the weirdo Tumblr-core mixtapes she used to release for free, linking with post-post-modern bubblegum bass crew PC Music"
   - Review B: "at times unapologetically poppy...But it's also utterly, defiantly weird, flouting conventions of rhythm, composition and, perhaps most of all, taste"

2. **Using distorted, glitched, or unconventional vocal processing**
   - Review A: "her humanoid vocals stutter and short-circuit"
   - Review B: "Like her Rochdale heroes, SOPHIE creates sounds with uncanny tactile qualities...wet and sticky one moment, metallic and electrified the next"

3. **Deliberately subverting expectations of what pop music should be**
   - Review A: "Instead, she doubled down on the weirdo Tumblr-core mixtapes...to try and figure out exactly what kind of artist she wan

### Experimenting with the recommender but only using reviews from a single publication

In [ ]:
# Pitchfork-only masked recommender — restricts both query and candidates to Pitchfork.
# More consistent vocabulary and review quality than the full mixed corpus.
pitchfork_mask = df["dataset"] == "pitchfork"
pitchfork_idx = df[pitchfork_mask].index.tolist()

def recommend_masked_pitchfork(query_album, query_artist=None, k=5, filter_same_artist=True):
    mask = (df["album_norm"] == query_album.lower().strip()) & pitchfork_mask
    if query_artist:
        mask &= df["artist_norm"] == query_artist.lower().strip()

    query_reviews = df[mask]
    if len(query_reviews) == 0:
        print(f"Not found in Pitchfork: '{query_album}'")
        return None

    query_artist_norm = query_reviews.iloc[0]["artist_norm"]
    q_text = query_reviews.iloc[0]["cleaned_text"]
    query_embedding = embeddings_masked[query_reviews.index].mean(axis=0)
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    sims = np.full(len(df), -np.inf)
    sims[pitchfork_idx] = embeddings_masked[pitchfork_idx] @ query_embedding
    sims[query_reviews.index] = -np.inf

    ranked = np.argsort(sims)[::-1]
    seen, results = set(), []
    for idx in ranked:
        artist_norm = df.loc[idx, "artist_norm"]
        if filter_same_artist and artist_norm == query_artist_norm:
            continue
        key = (artist_norm, df.loc[idx, "album_norm"])
        if key not in seen:
            seen.add(key)
            r_text = df.loc[idx, "cleaned_text"]
            results.append({
                "album":    df.loc[idx, "album"],
                "artist":   df.loc[idx, "artist"],
                "source":   df.loc[idx, "dataset"],
                "score":    round(float(sims[idx]), 4),
                "keywords": keyword_overlap(q_text, r_text),
                "snippet":  r_text[:200]
            })
        if len(results) >= k:
            break

    for i, r in enumerate(results, 1):
        print(f"\n{i}. {r['artist']} — {r['album']} ({r['source']}, score: {r['score']})")
        print(f"   Keywords: {', '.join(r['keywords'])}")
        print(f"   {r['snippet']}...")

    return results

recs = recommend_masked_pitchfork("kid a", query_artist="radiohead")


1. Aereogramme — Sleep and Release (pitchfork, score: 0.8413)
   Keywords: like, band, radiohead, useless, cue, guitar, begs, best
   Get ready, brothers and sisters and gender-assignees in the void, for a barrage of hyperbolic hypotheses: What if Radiohead got guitar-happy with U2-ish heft? What if The Flaming Lips stopped making n...

2. For Stars — ...It Falls Apart (pitchfork, score: 0.8349)
   Keywords: radiohead, album, like, band, belts, song, completely, maudlin
   If, for some reason, you're in search of a savior in these relatively fertile times for underground music, For Stars may be more than happy to oblige. ...It Falls Apart, the band's fourth full-length,...

3. Endochine — Day Two (pitchfork, score: 0.8298)
   Keywords: radiohead, yorke, thom, album, band, try, shooting, piano
   Little scribbly lines zigzagging and arcing over a black background, a TV with the message "I hate you" repeated on its screen, a little electrical schematic with "Endochine" scrawled in the m

In [ ]:
recs = recommend_masked_pitchfork("the archandroid", query_artist="janelle monáe")



1. Rhye — Woman (pitchfork, score: 0.7722)
   Keywords: songs, deliberately, comes, avoid, liner, album, music, pop
   Rhye's short history is marked by serendipity and mystery. A couple of years ago, after being tapped by Hannibal's Copenhagen-based electronic group Quadron, producer/vocalist Mike Milosh flew to Denm...

2. Victoria Monét — Jaguar (pitchfork, score: 0.772)
   Keywords: funk, resulting, way, songs, level, collaborators, story, star
   Victoria Monét was already in the studio when Ariana Grande meandered in, clutching Tiffany's bags, tipsy from champagne served at the store. The story behind how they wrote "7 rings," along with a sl...

3. Mhysa — NEVAEH (pitchfork, score: 0.7666)
   Keywords: fi, similar, inspirations, resonance, sentiments, world, ego, jackson
   When it comes to reinterpreting songs, how you sing matters just as much as what you're singing. Louis Armstrong turned an apocalyptic dirge into a celebration of new life and a New Orleans jazz funer...

4.

In [ ]:
sade_row = df[(df["artist_norm"] == "sade") & (df["album_norm"] == "diamond life") & (df["dataset"] == "pitchfork")].iloc[0]
monae_row = df[(df["artist_norm"] == "janelle monáe") & (df["album_norm"] == "the archandroid") & (df["dataset"] == "pitchfork")].iloc[0]

explain_pair_simple(
    monae_row["album_norm"], monae_row["artist_norm"],
    sade_row["album_norm"],  sade_row["artist_norm"],
    source="pitchfork"
)



  janelle monáe — the archandroid (pitchfork)
  → sade — diamond life (pitchfork)

1. **Distinctive approach to genre-blending within soul/R&B foundation**
   - Review A: "The songs zip gleefully from genre to genre, mostly grounded in R&B and funk, but spinning out into rap, pastoral British folk, psychedelic rock, disco, cabaret, cinematic scores"
   - Review B: "their sound liquified soul and jazz into new-school pop"

2. **Masterful restraint and cohesion despite stylistic ambition**
   - Review A: "The most impressive thing about The ArchAndroid isn't that it bounces between genres, but that it does so without compromising quality or cohesion"
   - Review B: "Sade became the minimalists, crafting quiet, vintage soul out of basic components" and "executors of spaciousness"

3. **Influential prototype for subsequent generations of artists**
   - Review A: "marrying the world-building possibilities of the concept album to the big tent genre-mutating pop of Michael Jackson and Prince

In [ ]:
recs = recommend_masked_pitchfork("channel orange", query_artist="frank ocean")



1. Cocaine 80s — The Flower of Life (pitchfork, score: 0.8033)
   Keywords: frank, orange, channel, ocean, believer, bap, pursuit, life
   Last year's praise of Frank Ocean's Channel Orange scarcely mentioned the album's star utility player, co-producer, and multi-instrumentalist Malay, or the song that Frank didn't write. Channel Orange...

2. Lambchop — This (Is What I Wanted to Tell You) (pitchfork, score: 0.7795)
   Keywords: bout, frank, ocean, wrote, album, time, song, plea
   There's a trick Frank Ocean returns to throughout Blonde. After smothering his voice with all kinds of effects for long stretches, he'll cut the switch and present his voice naked, and every time it's...

3. Adrian Orange & Her Band — Adrian Orange & Her Band (pitchfork, score: 0.7689)
   Keywords: orange, thinking, artistically, storied, inspires, sentence, won, song
   There are undoubtedly a slew of people that will be enamored with this new Adrian Orange record. Maybe they're already taken with his siz

In [ ]:
doheny_row = df[(df["artist_norm"] == "ned doheny") & (df["album_norm"] == "separate oceans") & (df["dataset"] == "pitchfork")].iloc[0]
ocean_row = df[(df["artist_norm"] == "frank ocean") & (df["album_norm"] == "channel orange") & (df["dataset"] == "pitchfork")].iloc[0]

explain_pair_simple(
    ocean_row["album_norm"], ocean_row["artist_norm"],
    doheny_row["album_norm"], doheny_row["artist_norm"],
    source="pitchfork"
)



  frank ocean — channel orange (pitchfork)
  → ned doheny — separate oceans (pitchfork)

1. **Gifted songwriting that deserves greater recognition than it received**
   - Review A: "he's got the type of voice, wit, charm, smarts, and ineffable humanity that's always hoped for, but never promised"
   - Review B: "There are no hits per se, but songs that should have—and likely could have—been hits"

2. **Emotional depth beneath a serene or polished surface**
   - Review A: "this serene deadpan is splashed with crackling emotion"
   - Review B: "The music sounds like it's been preserved in amber, so sharply redolent of the time and place of its creation"


### Testing a recommender that uses Nomic to select "finalist" albums that are then examined by an LLM (in this case, Claude Haiku) for best semantic match. Not cost-effective at scale but provides useful alternative niche option.

In [ ]:
# Full pipeline: masked recommender → LLM filter → ranked results with explanations.
# Gets candidate_k albums from masked embeddings, scores each with the explainer,
# returns only those with min_connections genuine quoted connections.
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

def recommend_with_llm_filter(query_album, query_artist=None,
                               candidate_k=10, return_k=3,
                               min_connections=3, pitchfork_only=True):

    # Step 1: get candidates from masked recommender
    if pitchfork_only:
        source_mask = pitchfork_mask
    else:
        source_mask = pd.Series([True] * len(df), index=df.index)

    mask = (df["album_norm"] == query_album.lower().strip()) & source_mask
    if query_artist:
        mask &= df["artist_norm"] == query_artist.lower().strip()

    query_reviews = df[mask]
    if query_reviews.empty:
        print(f"Not found: '{query_album}'")
        return None

    query_artist_norm = query_reviews.iloc[0]["artist_norm"]
    q_text   = query_reviews.iloc[0]["cleaned_text"]
    q_source = query_reviews.iloc[0]["dataset"]
    q_label  = f"{query_artist_norm} — {query_album} ({q_source})"

    query_embedding = embeddings_masked[query_reviews.index].mean(axis=0)
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    source_idx = df[source_mask].index.tolist()
    sims = np.full(len(df), -np.inf)
    sims[source_idx] = embeddings_masked[source_idx] @ query_embedding
    sims[query_reviews.index] = -np.inf

    ranked = np.argsort(sims)[::-1]
    seen, candidates = set(), []
    for idx in ranked:
        artist_norm = df.loc[idx, "artist_norm"]
        if artist_norm == query_artist_norm:
            continue
        key = (artist_norm, df.loc[idx, "album_norm"])
        if key not in seen:
            seen.add(key)
            candidates.append(idx)
        if len(candidates) >= candidate_k:
            break

    print(f"Scoring {len(candidates)} candidates with LLM filter...")

    # Step 2: run explainer on each candidate
    results = []
    for idx in candidates:
        r_text   = df.loc[idx, "cleaned_text"]
        r_source = df.loc[idx, "dataset"]
        r_label  = f"{df.loc[idx, 'artist']} — {df.loc[idx, 'album']} ({r_source})"

        prompt = f"""You are analyzing two music reviews to find their most meaningful shared qualities.

Identify up to 3 significant qualities that both albums genuinely share, based ONLY on what is written in the reviews.
If fewer than 3 genuine connections exist, list only those that are real.
If there are no meaningful connections, return exactly: NONE

For each shared quality:
- State it in one clear phrase
- Quote the specific language from each review that supports it (verbatim, in quotation marks, attributed to Review A or Review B)

Album A: {q_label}
Review A:
{q_text[:1500]}

Album B: {r_label}
Review B:
{r_text[:1500]}

Return a numbered list only. No introduction, no conclusion. If no genuine connections, return: NONE"""



        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=400,
            messages=[{"role": "user", "content": prompt}]
        )
        result = response.content[0].text.strip()

        if result == "NONE":
            n = 0
        else:
            n = len([l for l in result.split("\n") if l.strip().startswith(("1.", "2.", "3."))])

        if n >= min_connections:
            results.append({
                "album":       df.loc[idx, "album"],
                "artist":      df.loc[idx, "artist"],
                "source":      r_source,
                "score":       round(float(sims[idx]), 4),
                "connections": n,
                "explanation": result
            })

        if len(results) >= return_k:
            break

    # Step 3: print results
    if not results:
        print(f"No candidates met the minimum of {min_connections} genuine connections.")
        return None

    print(f"\nFound {len(results)} recommendations with {min_connections}+ genuine connections\n")
    for i, r in enumerate(results, 1):
        print(f"\n{'='*60}")
        print(f"{i}. {r['artist']} — {r['album']} ({r['source']}, score: {r['score']})")
        print(f"{'='*60}")
        print(f"\n{r['explanation']}")

    return results

# Test
recommend_with_llm_filter("channel orange", "frank ocean");


Scoring 10 candidates with LLM filter...

Found 3 recommendations with 3+ genuine connections


1. Lambchop — This (Is What I Wanted to Tell You) (pitchfork, score: 0.7795)

1. Strategic use of vocal effects and restraint
   - Review A: "this serene deadpan is splashed with crackling emotion"
   - Review B: "After smothering his voice with all kinds of effects for long stretches, he'll cut the switch and present his voice naked"

2. Emotional depth achieved through minimalist approach
   - Review A: "he's weathered it, and he's come away with his Zen-like calm intact"
   - Review B: "Wagner opts for tasteful restraint"

3. Sophisticated exploration of vulnerability within controlled artistic frameworks
   - Review A: "he's got the type of voice, wit, charm, smarts, and ineffable humanity that's always hoped for, but never promised"
   - Review B: "For an artist who came to electronic music so late in life, it's remarkable how astute his impulses are"

2. Gallant — Ology (pitchfork, sco